In [5]:
import os
root = "/lakehouse/default/Files/crimedata"
for dirpath, dirnames, filenames in os.walk(root):
    for name in filenames:
          print(os.path.join(dirpath, name))

##First, though, diagnose whether it's the recursion or the path itself — run just the top level:

import notebookutils
notebookutils.fs.ls(folder)   # does ONE level work?

##- If this works, the recursion is descending into a problematic subfolder (very commonly a OneLake shortcut whose target is broken/inaccessible — enumerating it throws this exact error).
##- If this also errors, it's the endpoint/path — usually transient; OneLake intermittently 500s here, and a retry often just works.

##If you want to stay on notebookutils.fs, make the recursion defensive so one bad subfolder (or a transient blip) doesn't kill the whole walk:

import notebookutils, time

def list_all(path, retries=2):
    for attempt in range(retries + 1):
        try:
            entries = notebookutils.fs.ls(path)
            break
        except Exception as e:
            if attempt == retries:
                print(f"SKIP {path} -> {e}")
                return
            time.sleep(1)  # transient OneLake 500 — back off and retry
    for f in entries:
        if f.isDir:
            yield from list_all(f.path)
        else:
            yield f.path

for p in list_all(folder):
    print(p)


/lakehouse/default/Files/crimedata/2020-01/2020-01-avon-and-somerset-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-bedfordshire-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-btp-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-cambridgeshire-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-cheshire-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-city-of-london-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-cleveland-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-cumbria-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-derbyshire-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-devon-and-cornwall-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-dorset-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-durham-street.csv
/lakehouse/default/Files/crimedata/2020-01/2020-01-dyfed-powys-street.csv
/lakehouse/default/Files/crimedata/2020-01/202

HttpResponseError: Data at the root level is invalid. Line 1, position 1.
ErrorCode:InternalServerError
Content: <?xml version="1.0" encoding="utf-8"?>
<Error>
  <Code>InternalServerError</Code>
  <Message>Data at the root level is invalid. Line 1, position 1.</Message>
</Error>